# Extremes Reanalysis from "The Extraordinary March 2022 East Antarctica Heat Wave. Part I: Observations and Meteorological Drivers"

To analyze the return times of this extreme event, the authors fit a Generalized Pareto Distribution (GPD) to the data past some extreme threshold. An estimated return time is then computed from the probability of observing another temperature as extreme or more extreme than what was observed in the heat wave, under this GPD fit. Uncertainty bounds, in the form of a confidence interval, is obtained from the parametric bootstrap.

A few issues I see with this:
1. The GPD is an approximate result, only exact in the limit as the threshold goes to infinity, and the sample size also goes to infinity. Since both are necessarily finite in an applied analysis, the GPD fit may not work very well and should be questioned. (sensitivity analysis with regards to threshold selection was mentioned, but results not reported or expanded upon)
2. Bootstrapping in the tail is generally frowned upon (bootstrapping requires that you have sufficient data such that your data looks like the actual distribution, but in extremes it is necessarily that we have little data to go off of and hence the performance of the bootstrap may not be so good)

I want to do my own little re-analysis here, testing sensitivity to different threshold levels as well as the uncertainty quantification scheme.

## Uncertainty Quantification Scheme Test

In [1]:
import scipy.stats as stats
import scipy.optimize as optim
import numpy as np
import matplotlib.pyplot as plt

In [2]:
daily_temp_obs = stats.norm.rvs(loc=-30, scale = 10, size=10000)

In [3]:
u = np.quantile(daily_temp_obs, 0.98)
data = daily_temp_obs[daily_temp_obs > u] - u

In [56]:
def gpd_negloglikelihood(params, data):
    shape = params[0]
    scale = params[1]
    terms = np.log((((data*shape/scale) + 1)**(-1/(1+shape)))/scale)
    
    return(-sum(terms))
    

In [81]:
optim.minimize(fun=gpd_negloglikelihood, x0=np.array([1/2,1/2]), method='SLSQP', args = (data), constraints=cons, bounds=bnds)

 message: Inequality constraints incompatible
 success: False
  status: 4
     fun: -2497.974105746167
       x: [ 1.000e-06  1.000e-06]
     nit: 2
     jac: [ 1.317e+08  6.731e+07]
    nfev: 6
    njev: 2

In [80]:
def constraint_func(x, data):
    return(min(data) + (x[1]/x[0]))
cons = ({'type':'ineq', 'fun':constraint_func, 'args':(data,)})

In [78]:
bnds = ((0.000001, None), (0.000001, None))